# The K-Suiter: patient-specific EEG channel ranking

Given a patient and the strict worst-patient `K` saved by K-Finder, this notebook ranks and returns the `K` EEG channels that best suit that patient's seizure-forecasting history. The latest seizure/control pair is reserved for a final untouched evaluation.

The ranking uses expanding chronological validation: models learn from earlier seizure episodes and validate on later episodes. At each step, every remaining channel is tested alongside the channels already selected. The candidate with the highest value is added:

$$\text{value} = 2\times\text{sensitivity} - 0.10\times\text{false alarms/hour} - 0.25\times\text{time in warning} + 0.20\times\text{AUROC}$$

The utility rewards seizure capture and AUROC while penalizing false alarms and time in warning, matching the rolling forecast alarm policy. Selection is repeated across expanding historical prefixes and records channel stability.

> **Research only:** this ranking has not been clinically validated and must not directly control patient care.

## 1. Inputs

Set the patient identifier. The channel count `K` is loaded automatically from the strict worst-patient handoff written by `k-finder.ipynb`, so it is never copied from a displayed notebook output. Siena identifiers look like `PN00`, `PN06`, and `PN10`.

In [1]:
PATIENT_ID = "PN00"
# K is read from results/sensor_count_selected.json, written by k-finder.ipynb.
FORCE_REBUILD_FEATURES = False

## 2. Setup

The notebook can be launched from either the scripts directory or the repository root. Feature extraction is cached; the first run for a patient may take several minutes.

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "k_suiter.py").exists():
    candidates = list(NOTEBOOK_DIR.rglob("k_suiter.py"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Run this notebook from the scripts directory or repository root."
        )
    NOTEBOOK_DIR = candidates[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

import rolling_seizure_forecasting as rsf
import personalized_channels_workflow as pc
from k_suiter import KSuiter, load_k_finder_result

paths = pc.personalized_paths(NOTEBOOK_DIR)
forecast_config = rsf.ForecastConfig(test_fraction=0.20, max_iter=60)
k_finder_result = load_k_finder_result(paths["project"])
K = k_finder_result.k
suiter_config = pc.PersonalizedConfig(
    k=K,
    patient_ids=(PATIENT_ID,),
    swap_refinement=False,
    force_rebuild_features=FORCE_REBUILD_FEATURES,
)
suiter_config.validate()
print(f"Patient={PATIENT_ID}; K={K} (from {k_finder_result.source_path.name})")

Patient=PN00; K=12 (from sensor_count_selected.json)


## 3. Load one patient's channel-local features

Eligibility requires at least two usable historical seizure events and at least `K` consistently available channels. Only channel-local features are used, so an excluded electrode cannot leak information into a selected electrode.

In [3]:
manifest = pc.load_manifest(paths, forecast_config)
available_patients = sorted(manifest["patient_id"].astype(str).unique())
if PATIENT_ID not in available_patients:
    raise ValueError(
        f"Unknown patient {PATIENT_ID!r}. Available patients: {available_patients}"
    )

patient_manifest = manifest.loc[
    manifest["patient_id"].astype(str).eq(PATIENT_ID)
].copy()
n_events = patient_manifest.loc[
    patient_manifest["episode_type"].eq("preictal"), "source_event_id"
].nunique()
if n_events < 2:
    raise ValueError(
        f"{PATIENT_ID} has {n_events} usable seizure event(s); at least 2 are required."
    )

patient = pc.build_patient_feature_data(
    patient_manifest,
    paths["feature_cache"],
    forecast_config,
    force=FORCE_REBUILD_FEATURES,
)
patient_overview = pd.DataFrame(
    {
        "patient_id": [patient.patient_id],
        "usable_seizure_events": [n_events],
        "available_channels": [len(patient.channel_names)],
        "landmark_rows": [len(patient.frame)],
    }
)
display(patient_overview)

PN00: loaded cached channel features.


,patient_id,usable_seizure_events,available_channels,landmark_rows
0,PN00,5,29,1500


## 4. Run the K-Suiter

The notebook runs the selected K-channel montage through the Gen1 two-stage forecasting model on the same patient-specific development/holdout split. The final table below is Gen1's held-out metric table; the notebook also writes the channel ranking audit, Gen1 metric CSVs, and a JSON handoff under `results/k_suiter`.

In [4]:
suiter = KSuiter(config=suiter_config)
selected_channels = suiter.recommend_from_k_finder(
    patient, paths["project"], manifest=patient_manifest
)
recommendation_json, ranking_csv = suiter.save_recommendation(
    paths["project"] / "results" / "k_suiter"
)

print(f"Top {K} channels for {PATIENT_ID}: {selected_channels}")
print(f"Saved model-ready channels: {recommendation_json}")
print(f"Saved ranking audit: {ranking_csv}")
display(suiter.gen1_metrics_.style.format(
    {
        "value": "{:.4f}",
    }
))

Top 12 channels for PN00: ['C3', 'F9', 'F7', 'FC6', 'FC2', 'PZ', 'O1', 'P3', 'F10', 'FZ', 'O2', 'F8']
Saved model-ready channels: C:\Users\sahil\Documents\cosmos\26-the-optimizers-analysis\final_project\results\k_suiter\k_suiter_PN00_k12_recommendation.json
Saved ranking audit: C:\Users\sahil\Documents\cosmos\26-the-optimizers-analysis\final_project\results\k_suiter\k_suiter_PN00_k12_ranking.csv


,metric,value,interpretation
0,negative log likelihood,1.1592,Lower is better; proper score for the observed bin/no-event class.
1,multicategory Brier score,0.3069,Lower is better; probability error across 60 bins plus no-event.
2,multicategory Brier skill score,0.1459,Above 0 improves on the development-set class-frequency forecast.
3,integrated survival Brier score,0.0683,Lower is better; mean survival-probability error across 5-minute horizon.
4,5-minute AUROC,0.9115,Discrimination only; does not assess calibration.
5,5-minute average precision,0.5314,Ranking metric sensitive to event prevalence.
6,conditional timing MAE (seconds),75.0000,Error of expected onset time among seizure landmarks.
7,seizure sensitivity,0.0000,Fraction of seizure episodes with an alarm after 18 consecutive high-risk landmarks.
8,time in warning,0.0000,Fraction of evaluated 5-second landmarks under warning.
9,false alarms per hour,0.0000,Rising persistent-alarm edges in interictal episodes per monitored hour.


## 5. Interpretation

- Rank 1 is the strongest channel when used alone.
- Each later channel is the best addition conditional on the channels above it.
- Validation is chronological and patient-specific; results should not be interpreted as a universal electrode ranking.
- A high validation score on a patient with few events is uncertain. Report the event count with every ranking.
- The selected channels should be evaluated on a later untouched episode before making research performance claims.